## Final Project: Collecting Your Corpus

This notebook builds the corpus for the final project, start to finish: it
takes the metadata table you compiled by hand, resolves each title to a real
Wikipedia article, collects every table the project description asks for,
annotates the text with spaCy, and saves the result as a directory of Parquet
files with a manifest and the requests cache that the tables were built from.
It is the skeleton of the *collection notebook* you will hand in. Read it as
you run it, change the settings so they describe your population, and add or
cut whatever your proposal calls for.

The cells are meant to be run in order, top to bottom. The first run reaches
the network and takes a while (several minutes for a corpus of 150 pages,
mostly on the revisions and the spaCy step). Every later run of the same cells
in the same session is answered from the cache and is fast. The notebook ends
by zipping the corpus directory and downloading it to your computer; that zip
is what you hand in, what the analysis notebook loads, and what you upload at
the top of this notebook to bring the cache back in a new session.

### What your metadata file needs to look like

Before you start, you need a CSV file with one row per page in your
population. It must have a column named `doc_id` holding the Wikipedia title
of each page, spelled the way you found it, and at least two columns you coded
yourself from a source that is not Wikipedia. Column names should be lowercase
with underscores instead of spaces. Here is the shape, using the Virginia
counties example from the project page:

```
doc_id,region,population_2020,independent_city
"Albemarle County, Virginia",Piedmont,112395,no
"Richmond, Virginia",Central,226610,yes
"Fairfax County, Virginia",Northern,1150309,no
```

The titles do not have to be perfect. The notebook follows redirects, so
`Charlotte Bronte` will resolve to `Charlotte Brontë`, and it reports every
title it had to fix and every title it could not find. If your population came
from a category or from Wikidata rather than a list, you still need this file:
export the titles your rule produced, add your hand-coded columns, and upload
the result.

### Setup

Run all of the following before starting. The first cell fetches the course
`funs` module, the second installs the two packages Colab does not ship with,
and the third downloads the spaCy English model. If your population is in
another language edition, change the model name in the settings below and
install the matching one here (for example `fr_core_news_sm` for French).

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
! pip install requests-cache beautifulsoup4 --quiet

In [ ]:
! python -m spacy download en_core_web_sm

In [ ]:
import io
import re
import shutil
import time
from datetime import date
from pathlib import Path
from urllib.parse import quote

import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs
from funs import DSText

import requests
import requests_cache
import spacy
from bs4 import BeautifulSoup

### Settings

Everything you decide lives in this one cell, so that the manifest at the end
can be written from it and so that a reader of your code finds the decisions
in one place rather than scattered through the loops. Change every value to
describe your project. The user agent must contain your real email address;
Wikimedia asks for a way to contact the person behind a script.

In [ ]:
# Which language edition of Wikipedia your population comes from.
LANG = "en"

# Your population rule, in one or two sentences. This goes in the manifest.
POPULATION_RULE = """
Describe the rule here: which language edition, which categories, source list,
or Wikidata criteria, whether subcategories were included, and what you did
with lists, disambiguations, and redirects.
""".strip()

# A real user agent with a real email address.
USER_AGENT = "DSST389Project/1.0 (yournetid@richmond.edu)"

# The fixed window for daily page views, as YYYYMMDD strings.
PV_START = "20250101"
PV_END = "20251231"

# Revisions per page: the most recent REV_CAP (at most 500), or None for the
# full history of every page.
REV_CAP = 500

# The spaCy model used to annotate the text.
SPACY_MODEL = "en_core_web_sm"

# The name of the directory that will hold the corpus, and of the zip file.
CORPUS_NAME = "corpus"

### Where the files live

Everything this notebook produces goes into one directory on the Colab disk:
the metadata file you upload, the requests cache, the Parquet tables, and the
manifest. Colab wipes that disk when the runtime disconnects, so the last
cells of the notebook zip the directory and download it to your computer.

If you already have a zip from an earlier session, set `RESTORE_FROM_ZIP` to
`True` and the cell below asks you to upload it. That brings back the cache
along with everything else, so the rerun reaches the network zero times, as
in Chapter 12. Otherwise leave it `False` and start from an empty directory.

In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

LOCAL_DIR = Path(CORPUS_NAME)
RESTORE_FROM_ZIP = False

if RESTORE_FROM_ZIP and IN_COLAB:
    uploaded = files.upload()
    name = next(iter(uploaded))
    shutil.unpack_archive(name, LOCAL_DIR, "zip")
    print(f"Restored {name} into {LOCAL_DIR}:")
    for p in sorted(LOCAL_DIR.iterdir()):
        print("  ", p.name)

LOCAL_DIR.mkdir(parents=True, exist_ok=True)

### Upload your metadata

The first time through, the cell below opens a file chooser; pick your
metadata CSV. The file is saved into the corpus directory under the name
`metadata.csv`, exactly as uploaded, so that it travels with the corpus and
so that every later run reads it from disk instead of asking you to upload it
again. To replace it with a new version, set `REUPLOAD` to `True` for one run.

In [ ]:
META_PATH = LOCAL_DIR / "metadata.csv"
REUPLOAD = False

if REUPLOAD or not META_PATH.exists():
    if not IN_COLAB:
        raise FileNotFoundError(f"Put your metadata CSV at {META_PATH} first.")
    uploaded = files.upload()
    name, content = next(iter(uploaded.items()))
    META_PATH.write_bytes(content)
    print(f"Saved {name} as {META_PATH}")

meta = pl.read_csv(META_PATH)
meta

A few checks before anything goes over the network. The `doc_id` column must
exist and be complete, the titles must be unique, and the population should
fall in the 75 to 250 range the project asks for. The last of these is a
warning rather than an error, because the rule is the thing to fix, not the
check.

In [ ]:
if "doc_id" not in meta.columns:
    raise ValueError(f"No column named 'doc_id'; columns are {meta.columns}")

n_null = meta["doc_id"].null_count()
n_dup = len(meta) - meta["doc_id"].n_unique()
if n_null or n_dup:
    raise ValueError(f"{n_null} missing titles and {n_dup} duplicated titles")

if not (75 <= len(meta) <= 250):
    print(f"Warning: {len(meta)} pages is outside the 75 to 250 range.")

meta_columns = [x for x in meta.columns if x != "doc_id"]
print(f"{len(meta)} pages, {len(meta_columns)} metadata columns: {meta_columns}")

### The cache and a polite client

This is the setup from Chapter 12, unchanged except that the cache file sits
inside the corpus directory, so that it is saved and restored with everything
else. Successful responses and 404s are both cached and never expire.

In [ ]:
session = requests_cache.CachedSession(
    cache_name=str(LOCAL_DIR / "requests_cache"),
    backend="sqlite",
    allowable_methods=("GET", "HEAD", "POST"),
    allowable_codes=(200, 404),
    expire_after=None
)

base_url = f"https://{LANG}.wikipedia.org/w/api.php"
headers = {"User-Agent": USER_AGENT}


def polite_get(url, params=None):
    session.cookies.clear()
    response = session.get(url, params=params, headers=headers)
    response.raise_for_status()
    if not getattr(response, "from_cache", False):
        time.sleep(0.2)
    return response

### Resolving the titles

The spine of the corpus is the `pages` table, keyed on `doc_id`: the resolved
article title, which is the string every other table will carry. The `doc_id`
you typed in the metadata file becomes `source_title`, kept so that a reader
can trace each row back to that file when a redirect changed the name. The
function is the one from the end of Chapter 12; it
sends fifty titles per request with `redirects=1` and follows whatever
normalizations and redirects the server reports.

In [ ]:
def resolve_titles(titles):
    rows = []
    batches = [titles[i:(i + 50)] for i in range(0, len(titles), 50)]
    for batch in batches:
        params = {
            "action": "query",
            "format": "json",
            "prop": "info",
            "redirects": 1,
            "titles": "|".join(batch)
        }
        data = polite_get(base_url, params).json()

        followed = {}
        for kind in ("normalized", "redirects"):
            for step in data["query"].get(kind, []):
                followed[step["from"]] = step["to"]

        pages = {p["title"]: p for p in data["query"]["pages"].values()}
        for title in batch:
            final = title
            while final in followed:
                final = followed[final]
            rows.append({
                "source_title": title,
                "doc_id": final,
                "page_id": pages.get(final, {}).get("pageid")
            })
    return pl.DataFrame(rows)


resolved = resolve_titles(meta["doc_id"].to_list())
resolved

Two lists to read carefully. Titles that moved are usually fine (a redirect
from a short name to the full one), but check that each one landed where you
meant it to. Titles that resolved to nothing have no article and no redirect;
they are dropped from the corpus below, and the manifest records them. Fix
the typos in your CSV and re-upload, or keep the omission and say so in the
datasheet.

In [ ]:
moved = resolved.filter(c.source_title != c.doc_id)
missing = resolved.filter(c.page_id.is_null())

print(f"{len(moved)} titles followed a redirect or normalization:")
print(moved)
print(f"\n{len(missing)} titles could not be found:")
print(missing)

Now the spine itself: your metadata joined to the resolution table, with the
missing pages dropped. Two typed titles can resolve to the same article (an
author listed under two names, say); the `unique` call keeps the first and
the line after it tells you if that happened.

In [ ]:
pages = (
    meta
    .rename({"doc_id": "source_title"})
    .join(resolved, on=c.source_title)
    .filter(c.page_id.is_not_null())
    .unique(subset="doc_id", keep="first", maintain_order=True)
    .select(c.doc_id, c.page_id, c.source_title, pl.exclude("doc_id", "page_id", "source_title"))
    .sort(c.doc_id)
)

n_collapsed = len(resolved.filter(c.page_id.is_not_null())) - len(pages)
if n_collapsed:
    print(f"Warning: {n_collapsed} rows collapsed onto an article already in the corpus.")
print(f"{len(pages)} pages in the corpus")
pages

### Coordinates

Pages about places (and a surprising number of pages about other things)
carry a primary geographic coordinate, which the `coordinates` property
returns in batches of fifty. Pages without one simply have no entry, so the
columns are null for them. The coordinates are attached to the `pages` table
rather than kept separately, because there is at most one pair per page.

In [ ]:
ids = pages["page_id"].to_list()
batches = [ids[i:(i + 50)] for i in range(0, len(ids), 50)]

rows = []
for batch in batches:
    params = {
        "action": "query",
        "format": "json",
        "prop": "coordinates",
        "coprimary": "primary",
        "colimit": "max",
        "pageids": "|".join(str(i) for i in batch)
    }
    data = polite_get(base_url, params).json()
    for page in data["query"]["pages"].values():
        for coord in page.get("coordinates", [])[:1]:
            rows.append({
                "page_id": page["pageid"],
                "lat": coord["lat"],
                "lon": coord["lon"]
            })

coords = pl.DataFrame(rows, schema={"page_id": pl.Int64, "lat": pl.Float64, "lon": pl.Float64})
pages = pages.join(coords, on=c.page_id, how="left")

print(f"{len(coords)} of {len(pages)} pages have coordinates")
pages.select(c.doc_id, c.lat, c.lon).filter(c.lat.is_not_null()).head()

### Texts

Chapter 12 took the article text from the TextExtracts property, which
returns a plain-text rendering of the whole page, headings and all. Here we
take a stricter view of what counts as the text of an article. The `parse`
action returns the rendered HTML of the page, and we keep only what sits
inside paragraph tags, with every other tag stripped away. That drops the
infobox, the tables, the image captions, the section headings, the lists of
works, and the reference section, all of which are real parts of the page but
none of which are prose. Footnote markers such as `[12]` are removed as well,
since they would otherwise appear in the token table as punctuation around a
number.

This is a decision, and the datasheet should say that it was made. A page
whose content is mostly a table or a list will come out short here even if
the page itself is long.

In [ ]:
def paragraph_text(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup.select("sup.reference, sup.noprint, span.mw-editsection, style, script"):
        tag.decompose()
    paras = []
    for p in soup.find_all("p"):
        text = re.sub(r"\s+", " ", p.get_text()).strip()
        if text:
            paras.append(text)
    return "\n\n".join(paras)


rows = []
for row in pages.iter_rows(named=True):
    params = {
        "action": "parse",
        "format": "json",
        "prop": "text",
        "disableeditsection": 1,
        "disabletoc": 1,
        "pageid": row["page_id"]
    }
    data = polite_get(base_url, params).json()
    html = data["parse"]["text"]["*"]
    rows.append({"doc_id": row["doc_id"], "text": paragraph_text(html)})

texts = (
    pl.DataFrame(rows)
    .with_columns(n_chars = c.text.str.len_chars())
)
texts

Look at the shortest pages before going on. A corpus where a quarter of the
texts are a few hundred characters long will not support the textual
chapters, and this is the moment to find that out.

In [ ]:
print(texts.row(0, named=True)["text"][:500])
print()
texts.sort(c.n_chars).select(c.doc_id, c.n_chars).head(10)

### Links

One row per outgoing internal link, in full, exactly as in Chapter 12. The
network chapter will filter this down to the edges whose target is also in
the corpus; that view is derived from this table, not stored in its place.

In [ ]:
rows = []
for row in pages.iter_rows(named=True):
    params = {
        "action": "query",
        "format": "json",
        "prop": "links",
        "plnamespace": 0,
        "pllimit": "max",
        "pageids": row["page_id"]
    }
    while True:
        data = polite_get(base_url, params).json()
        page = data["query"]["pages"][str(row["page_id"])]
        for link in page.get("links", []):
            rows.append({"doc_id": row["doc_id"], "link_title": link["title"]})
        if "continue" not in data:
            break
        params.update(data["continue"])

links = pl.DataFrame(rows)
links

A quick check that the network is not empty: how many of those links point at
another page in the corpus, and how many pages have at least one such edge.

In [ ]:
links_internal = (
    links
    .join(pages.select(link_title = c.doc_id, to_doc_id = c.doc_id), on=c.link_title)
)

print(f"{len(links_internal)} internal edges among {len(links)} links")
print(f"{links_internal['doc_id'].n_unique()} of {len(pages)} pages link to another page in the corpus")

### Revisions

The revision history, capped at the `REV_CAP` most recent edits per page
(500 is the most one request returns). Setting `REV_CAP` to `None` follows
the `continue` token to the beginning of each page's history, which is the
right choice only when your corpus is small and a question needs it. Either
way, say which you did in the manifest; the cell below writes it in for you.
The timestamps are parsed into a real datetime type here, so the temporal
chapter can start work immediately.

In [ ]:
rows = []
for row in pages.iter_rows(named=True):
    params = {
        "action": "query",
        "format": "json",
        "prop": "revisions",
        "rvprop": "ids|timestamp|user|size|comment",
        "rvlimit": "max" if REV_CAP is None else REV_CAP,
        "pageids": row["page_id"]
    }
    while True:
        data = polite_get(base_url, params).json()
        page = data["query"]["pages"][str(row["page_id"])]
        for rev in page.get("revisions", []):
            rows.append({
                "doc_id": row["doc_id"],
                "rev_id": rev["revid"],
                "timestamp": rev["timestamp"],
                "user": rev.get("user"),
                "size": rev["size"],
                "comment": rev.get("comment")
            })
        if REV_CAP is not None or "continue" not in data:
            break
        params.update(data["continue"])

revisions = (
    pl.DataFrame(rows)
    .with_columns(
        timestamp = c.timestamp.str.to_datetime("%Y-%m-%dT%H:%M:%SZ")
    )
)
revisions

### Page views

Daily views over the fixed window from the settings cell, from the Wikimedia
REST API, with the `user` agent class so that bots are excluded. A page with
no view data answers with a 404, which the cache records and this loop
treats as information: the page is listed below and has no rows in the table.

In [ ]:
pv_base = (
    "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
    f"{LANG}.wikipedia/all-access/user/{{article}}/daily/{PV_START}/{PV_END}"
)

rows = []
no_views = []
for row in pages.iter_rows(named=True):
    article = quote(row["doc_id"].replace(" ", "_"), safe="")
    session.cookies.clear()
    response = session.get(pv_base.format(article=article), headers=headers)
    if not getattr(response, "from_cache", False):
        time.sleep(0.2)
    if response.status_code == 404:
        no_views.append(row["doc_id"])
        continue
    response.raise_for_status()
    for item in response.json()["items"]:
        rows.append({
            "doc_id": row["doc_id"],
            "date": item["timestamp"],
            "views": item["views"]
        })

pageviews = (
    pl.DataFrame(rows)
    .with_columns(date = c.date.str.slice(0, 8).str.to_date("%Y%m%d"))
)

print(f"{len(no_views)} pages without view data: {no_views}")
pageviews

### Language links

Which other language editions have an article for each page, and under what
title, fifty pages per request.

In [ ]:
ids = pages["page_id"].to_list()
batches = [ids[i:(i + 50)] for i in range(0, len(ids), 50)]

rows = []
for batch in batches:
    params = {
        "action": "query",
        "format": "json",
        "prop": "langlinks",
        "lllimit": "max",
        "pageids": "|".join(str(i) for i in batch)
    }
    while True:
        data = polite_get(base_url, params).json()
        for page in data["query"]["pages"].values():
            for link in page.get("langlinks", []):
                rows.append({
                    "page_id": page["pageid"],
                    "lang": link["lang"],
                    "lang_title": link["*"]
                })
        if "continue" not in data:
            break
        params.update(data["continue"])

langlinks = (
    pl.DataFrame(rows, schema={"page_id": pl.Int64, "lang": pl.String, "lang_title": pl.String})
    .join(pages.select(c.page_id, c.doc_id), on=c.page_id)
    .select(c.doc_id, c.lang, c.lang_title)
    .sort(c.doc_id, c.lang)
)
langlinks

### Annotations

The last table is the one Chapter 14 built in its first section: every
document through the spaCy pipeline, one row per token. It is built from the
`texts` table and saved alongside it, so that the analysis notebook can load
it in a second instead of spending several minutes re-running spaCy. The
`max_length` setting lifts spaCy's default limit on the size of a single
document, which a long article can exceed.

This cell is the slowest in the notebook, on the order of a minute per fifty
pages, and nothing about it is cached. Run it once, save the result, and do
not run it again unless the texts change.

In [ ]:
nlp = spacy.load(SPACY_MODEL)
nlp.max_length = 3_000_000

anno = DSText.process(texts.select(c.doc_id, c.text), nlp)
anno

In [ ]:
(
    anno
    .filter(c.upos == "NOUN")
    .group_by(c.lemma)
    .agg(count = pl.len())
    .sort(c.count, descending=True)
    .head(10)
)

### Saving the tables and the manifest

Everything goes into the corpus directory as Parquet, next to the cache and
the metadata file you uploaded. The manifest is written from the settings
and the counts, so that it cannot drift from what was actually collected.
Two lines of it are left for you to fill in by hand: where each of your
metadata columns came from, and how you resolved the ambiguous cases. Edit
the text in the cell rather than the file, so that a rerun does not overwrite
what you wrote.

In [ ]:
pages.write_parquet(LOCAL_DIR / "pages.parquet")
texts.write_parquet(LOCAL_DIR / "texts.parquet")
links.write_parquet(LOCAL_DIR / "links.parquet")
revisions.write_parquet(LOCAL_DIR / "revisions.parquet")
pageviews.write_parquet(LOCAL_DIR / "pageviews.parquet")
langlinks.write_parquet(LOCAL_DIR / "langlinks.parquet")
anno.write_parquet(LOCAL_DIR / "anno.parquet")

for name, df in [("pages", pages), ("texts", texts), ("links", links),
                 ("revisions", revisions), ("pageviews", pageviews),
                 ("langlinks", langlinks), ("anno", anno)]:
    print(f"{name:10s} {df.shape[0]:>9,} rows  {df.shape[1]:>3} columns")

In [ ]:
rev_note = (
    "full history of every page" if REV_CAP is None
    else f"the {REV_CAP} most recent revisions per page"
)

manifest = f"""# Wikipedia corpus: {CORPUS_NAME}

Retrieved: {date.today().isoformat()}
Sources: MediaWiki Action API ({LANG}.wikipedia.org/w/api.php) and
  Wikimedia REST API v1 (wikimedia.org/api/rest_v1)
Population: {POPULATION_RULE}
Pages: {len(pages)} (from {len(meta)} titles in metadata.csv;
  {len(missing)} not found and dropped: {missing["source_title"].to_list()};
  {len(moved)} followed a redirect or normalization, see source_title)
Texts: paragraph (<p>) text of the rendered page, other tags stripped,
  footnote markers removed
Revisions: {len(revisions)} rows, {rev_note}
Pageviews: {PV_START} to {PV_END}, daily, all-access, user agent class;
  {len(no_views)} pages with no view data: {no_views}
Coordinates: {len(coords)} of {len(pages)} pages carry a primary coordinate
Annotations: spaCy {spacy.__version__}, model {SPACY_MODEL}
Metadata columns (hand-coded, not from the API): {meta_columns}
  Source for each column: FILL IN
  How ambiguous cases were resolved: FILL IN
Tables: pages, texts, links, revisions, pageviews, langlinks, anno
Raw responses: requests_cache.sqlite
Uploaded metadata: metadata.csv
"""

(LOCAL_DIR / "MANIFEST.md").write_text(manifest)
print(manifest)

### Downloading the corpus

The last cell zips the whole directory, cache and metadata file included, and
downloads it to your computer. Keep that file somewhere safe. It is what you
upload to Box when you hand the project in, what the analysis notebook loads,
and what you upload at the top of this notebook if you ever need to run the
collection again in a new session.

In [ ]:
zip_path = shutil.make_archive(CORPUS_NAME, "zip", LOCAL_DIR)
print(zip_path)
if IN_COLAB:
    files.download(zip_path)

### Loading the corpus in the analysis notebook

The analysis notebook never calls the APIs. It asks for the zip, unpacks it,
and reads the Parquet files, which is the whole of its setup. The cell below
is the one it uses; the `project_analysis` notebook on the course page
already has it in place.

In [ ]:
from pathlib import Path
import shutil
import polars as pl
from polars import col as c
from google.colab import files

CORPUS_DIR = Path("corpus")
if not (CORPUS_DIR / "pages.parquet").exists():
    uploaded = files.upload()
    shutil.unpack_archive(next(iter(uploaded)), CORPUS_DIR, "zip")

pages = pl.read_parquet(CORPUS_DIR / "pages.parquet")
texts = pl.read_parquet(CORPUS_DIR / "texts.parquet")
links = pl.read_parquet(CORPUS_DIR / "links.parquet")
revisions = pl.read_parquet(CORPUS_DIR / "revisions.parquet")
pageviews = pl.read_parquet(CORPUS_DIR / "pageviews.parquet")
langlinks = pl.read_parquet(CORPUS_DIR / "langlinks.parquet")
anno = pl.read_parquet(CORPUS_DIR / "anno.parquet")

print((CORPUS_DIR / "MANIFEST.md").read_text())